# CS293 Final: Fine-Tuning Llama 3.2 1B for Prerequisite (Building On) Prediction

**Task:** Given a K-12 math problem, predict which CCSS standards are **prerequisites** the student needs before tackling the problem.

**Approach:** Fine-tune Llama 3.2 1B using QLoRA on the MathFish dataset, using "Building On" labels.

**Pipeline:**
1. Load & prepare MathFish data (filter to Building On labels)
2. Evaluate base model (pre-fine-tune) on held-out test set
3. Fine-tune with Unsloth + QLoRA
4. Evaluate fine-tuned model on same test set
5. Compare pre vs post accuracy

## 0. Install Dependencies

In [ ]:
%%capture
# Pin transformers to avoid the additional_chat_templates bug in 4.57+
!pip install "transformers>=4.51.0,<4.57.0" 
!pip install unsloth
!pip install --upgrade datasets

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Load MathFish Dataset

In [ ]:
import json
import requests
from collections import Counter

def load_mathfish_jsonl(split):
    url = f"https://huggingface.co/datasets/allenai/mathfish/resolve/main/{split}.jsonl"
    rows = []
    resp = requests.get(url, stream=True)
    for line in resp.iter_lines():
        if line:
            rows.append(json.loads(line))
    return rows

print("Loading train split...")
train_raw = load_mathfish_jsonl("train")
print(f"Train: {len(train_raw)} problems")

print("Loading test split...")
test_raw = load_mathfish_jsonl("test")
print(f"Test: {len(test_raw)} problems")

In [ ]:
# Check Building On label availability
from collections import Counter

def check_split(data, name):
    empty_standards = sum(1 for r in data if not r["standards"])
    has_building_on = sum(1 for r in data if any(rel == "Building On" for rel, code in r["standards"]))
    rel_types = Counter(rel for r in data for rel, code in r["standards"])
    print(f"\n--- {name} split ---")
    print(f"  Total problems: {len(data)}")
    print(f"  Empty standards: {empty_standards} ({empty_standards/len(data)*100:.1f}%)")
    print(f"  Has 'Building On' labels: {has_building_on} ({has_building_on/len(data)*100:.1f}%)")
    print(f"  Relation types: {dict(rel_types)}")

check_split(train_raw, "Train")
check_split(test_raw, "Test")

## 2. Prepare Data for Instruction Tuning (Building On)

In [ ]:
def format_example(row):
    """Format a MathFish row as an instruction-tuning pair.

    Input: problem text
    Output: comma-separated list of CCSS prerequisite codes (Building On)
    """
    problem_text = row["text"].strip()

    # Get only 'Building On' standards as ground truth
    building_on = sorted(set(
        code for rel_type, code in row["standards"]
        if rel_type == "Building On"
    ))

    if not building_on:
        return None  # Skip problems with no Building On labels

    standards_str = ", ".join(building_on)

    return {
        "id": row["id"],
        "instruction": (
            "You are a math education expert. Given a K-12 math problem, predict which "
            "Common Core State Standards (CCSS) are prerequisites that a student needs "
            "to already know before attempting this problem. These are the 'Building On' standards. "
            "Return ONLY the prerequisite standard codes as a comma-separated list (e.g., 3.OA.A.1, 2.NBT.B.5). "
            "Do not include any explanation."
        ),
        "input": problem_text,
        "output": standards_str,
        "labels": building_on,
    }

# Process train and test
train_data = [x for x in (format_example(r) for r in train_raw) if x is not None]
test_data = [x for x in (format_example(r) for r in test_raw) if x is not None]

print(f"Train examples (with Building On labels): {len(train_data)}")
print(f"Test examples (with Building On labels): {len(test_data)}")
print(f"\nSample train output: {train_data[0]['output']}")

In [ ]:
# Distribution of number of prerequisites per problem
label_counts = Counter(len(d["labels"]) for d in train_data)
print("Prerequisites per problem (train):")
for k in sorted(label_counts):
    print(f"  {k} prereqs: {label_counts[k]} problems ({label_counts[k]/len(train_data)*100:.1f}%)")

In [ ]:
# Format as chat templates for Llama 3.2 Instruct
def to_chat_format(example):
    return {
        "conversations": [
            {"role": "system", "content": example["instruction"]},
            {"role": "user", "content": example["input"]},
            {"role": "assistant", "content": example["output"]},
        ]
    }

train_chat = [to_chat_format(d) for d in train_data]
print(f"Formatted {len(train_chat)} training conversations")
print(f"\nSample conversation:")
for msg in train_chat[0]["conversations"]:
    print(f"  [{msg['role']}]: {msg['content'][:120]}..." if len(msg['content']) > 120 else f"  [{msg['role']}]: {msg['content']}")

## 3. Load Base Model & Evaluate Pre-Fine-Tune

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="meta-llama/Llama-3.2-1B-Instruct",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
print("Base model loaded.")

In [ ]:
import re
from tqdm import tqdm

def extract_standards_from_text(text):
    pattern = r'\b(?:K|[1-8]|N|A|F|G|S|HS)[.-][A-Z]{1,4}(?:[.-][A-Z])?(?:[.-]\d+[a-z]?)\b'
    found = re.findall(pattern, text)
    normalized = list(dict.fromkeys(found))
    return normalized

def predict_prerequisites(model, tokenizer, problem_text, max_new_tokens=128):
    """Run inference on a single problem and return predicted prerequisite codes."""
    messages = [
        {"role": "system", "content": (
            "You are a math education expert. Given a K-12 math problem, predict which "
            "Common Core State Standards (CCSS) are prerequisites that a student needs "
            "to already know before attempting this problem. These are the 'Building On' standards. "
            "Return ONLY the prerequisite standard codes as a comma-separated list (e.g., 3.OA.A.1, 2.NBT.B.5). "
            "Do not include any explanation."
        )},
        {"role": "user", "content": problem_text},
    ]

    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    return extract_standards_from_text(generated), generated

# Quick sanity check
FastLanguageModel.for_inference(model)
pred, raw = predict_prerequisites(model, tokenizer, test_data[0]["input"])
print(f"Problem: {test_data[0]['input'][:200]}...")
print(f"Ground truth prereqs: {test_data[0]['labels']}")
print(f"Predicted: {pred}")
print(f"Raw output: {raw}")

In [ ]:
def compute_metrics(predictions, ground_truths):
    def to_cluster(code):
        parts = code.split(".")
        if len(parts) >= 3:
            return ".".join(parts[:3])
        return code

    def to_domain(code):
        parts = code.split(".")
        if len(parts) >= 2:
            return ".".join(parts[:2])
        return code

    levels = {
        "standard": lambda x: x,
        "cluster": to_cluster,
        "domain": to_domain,
    }

    results = {}
    for level_name, transform in levels.items():
        total_precision = 0
        total_recall = 0
        total_f1 = 0
        exact_matches = 0
        n = len(predictions)

        for pred, gt in zip(predictions, ground_truths):
            pred_set = set(transform(c) for c in pred)
            gt_set = set(transform(c) for c in gt)

            if not pred_set and not gt_set:
                total_precision += 1
                total_recall += 1
                total_f1 += 1
                exact_matches += 1
                continue

            if pred_set and gt_set:
                tp = len(pred_set & gt_set)
                p = tp / len(pred_set)
                r = tp / len(gt_set)
                f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
            elif pred_set:
                p, r, f1 = 0, 0, 0
            else:
                p, r, f1 = 0, 0, 0

            total_precision += p
            total_recall += r
            total_f1 += f1
            if pred_set == gt_set:
                exact_matches += 1

        results[level_name] = {
            "precision": total_precision / n,
            "recall": total_recall / n,
            "f1": total_f1 / n,
            "exact_match": exact_matches / n,
        }

    return results

def print_metrics(results, title=""):
    if title:
        print(f"\n{'='*60}")
        print(f"  {title}")
        print(f"{'='*60}")
    print(f"{'Level':<12} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Exact Match':>12}")
    print("-" * 56)
    for level in ["standard", "cluster", "domain"]:
        m = results[level]
        print(f"{level:<12} {m['precision']:>10.3f} {m['recall']:>10.3f} {m['f1']:>10.3f} {m['exact_match']:>12.3f}")

In [ ]:
import random

random.seed(42)
eval_data = random.sample(test_data, min(200, len(test_data)))
print(f"Evaluating base Llama 3.2 1B on {len(eval_data)} test problems (Building On)...")

FastLanguageModel.for_inference(model)
base_predictions = []
base_raw_outputs = []

for ex in tqdm(eval_data):
    pred, raw = predict_prerequisites(model, tokenizer, ex["input"])
    base_predictions.append(pred)
    base_raw_outputs.append(raw)

ground_truths = [ex["labels"] for ex in eval_data]
base_results = compute_metrics(base_predictions, ground_truths)
print_metrics(base_results, "BASE MODEL — Prerequisites (Pre-Fine-Tune)")

## 4. Fine-Tune with QLoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.print_trainable_parameters()

In [ ]:
from datasets import Dataset
from unsloth.chat_templates import get_chat_template, standardize_sharegpt

tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

train_dataset = Dataset.from_list(train_chat)
train_dataset = standardize_sharegpt(train_dataset)

def apply_template(examples):
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
             for c in examples["conversations"]]
    return {"text": texts}

train_dataset = train_dataset.map(apply_template, batched=True)
print(f"Training dataset: {len(train_dataset)} examples")
print(f"\nSample formatted text (first 500 chars):")
print(train_dataset[0]["text"][:500])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        warmup_steps=50,
        num_train_epochs=8,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs_building_on",
        save_strategy="epoch",
        report_to="none",
    ),
)

print("Trainer ready. Starting fine-tuning...")

In [ ]:
trainer_stats = trainer.train()
print(f"\nTraining complete!")
print(f"Total training time: {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.4f}")

## 5. Evaluate Fine-Tuned Model

In [ ]:
print(f"Evaluating fine-tuned Llama 3.2 1B on {len(eval_data)} test problems (Building On)...")

FastLanguageModel.for_inference(model)
ft_predictions = []
ft_raw_outputs = []

for ex in tqdm(eval_data):
    pred, raw = predict_prerequisites(model, tokenizer, ex["input"])
    ft_predictions.append(pred)
    ft_raw_outputs.append(raw)

ft_results = compute_metrics(ft_predictions, ground_truths)
print_metrics(ft_results, "FINE-TUNED MODEL — Prerequisites (Post-Fine-Tune)")

## 6. Comparison: Pre vs Post Fine-Tuning (Building On)

In [ ]:
import pandas as pd

rows = []
for level in ["standard", "cluster", "domain"]:
    for model_name, res in [("1B base (zero-shot)", base_results), ("1B fine-tuned (Building On)", ft_results)]:
        rows.append({
            "Model": model_name,
            "Level": level,
            "Precision": round(res[level]["precision"], 3),
            "Recall": round(res[level]["recall"], 3),
            "F1": round(res[level]["f1"], 3),
            "Exact Match": round(res[level]["exact_match"], 3),
        })

df = pd.DataFrame(rows)
print("\n" + "=" * 80)
print("  COMPARISON: Base vs Fine-Tuned — Prerequisite Prediction")
print("=" * 80)
print(df.to_string(index=False))

print("\n\nF1 Improvement:")
for level in ["standard", "cluster", "domain"]:
    base_f1 = base_results[level]["f1"]
    ft_f1 = ft_results[level]["f1"]
    delta = ft_f1 - base_f1
    print(f"  {level:<12}: {base_f1:.3f} -> {ft_f1:.3f} (Δ = {delta:+.3f})")

## 7. Error Analysis

In [ ]:
def analyze_errors(predictions, ground_truths, eval_data):
    exact_correct = []
    partial_matches = []
    total_misses = []

    for i, (pred, gt) in enumerate(zip(predictions, ground_truths)):
        pred_set = set(pred)
        gt_set = set(gt)

        if pred_set == gt_set:
            exact_correct.append(i)
        elif pred_set & gt_set:
            partial_matches.append(i)
        else:
            total_misses.append(i)

    print(f"Exact match: {len(exact_correct)} ({len(exact_correct)/len(predictions)*100:.1f}%)")
    print(f"Partial match: {len(partial_matches)} ({len(partial_matches)/len(predictions)*100:.1f}%)")
    print(f"Total miss: {len(total_misses)} ({len(total_misses)/len(predictions)*100:.1f}%)")

    print(f"\n--- Sample Total Misses ---")
    for idx in total_misses[:5]:
        print(f"  Problem: {eval_data[idx]['input'][:100]}...")
        print(f"  GT prereqs: {ground_truths[idx]}, Pred: {predictions[idx]}")

print("BASE MODEL ERRORS (Building On):")
print("-" * 40)
_ = analyze_errors(base_predictions, ground_truths, eval_data)

print("\n\nFINE-TUNED MODEL ERRORS (Building On):")
print("-" * 40)
_ = analyze_errors(ft_predictions, ground_truths, eval_data)

## 8. Save Model

In [ ]:
model.save_pretrained("llama-1b-prereqs-lora")
tokenizer.save_pretrained("llama-1b-prereqs-lora")
print("LoRA adapters saved to llama-1b-prereqs-lora/")

import json
results_out = {
    "task": "Building On (Prerequisites)",
    "base_model": base_results,
    "finetuned_model": ft_results,
    "num_eval": len(eval_data),
    "num_train": len(train_data),
}

with open("evaluation_results_building_on.json", "w") as f:
    json.dump(results_out, f, indent=2)
print("Results saved to evaluation_results_building_on.json")